In [ ]:
import subprocess, sys
def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("llmcompressor", "compressed-tensors",
    "transformers>=4.45", "accelerate", "datasets")

import os, gc, time, json, math
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

assert torch.cuda.is_available(), \
    "Enable a GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0),
      "| CUDA:", torch.version.cuda,
      "| torch:", torch.__version__)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

WORKDIR = Path("/content/quant_lab"); WORKDIR.mkdir(exist_ok=True)
os.chdir(WORKDIR)

def free_mem():
    gc.collect(); torch.cuda.empty_cache()

def dir_size_gb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1e9

def time_generation(model, tok, prompt, max_new_tokens=64):
    """Greedy decode; reports latency & tokens/sec after a brief warmup."""
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    _ = model.generate(**inputs, max_new_tokens=4, do_sample=False)
    torch.cuda.synchronize()
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tok.eos_token_id)
    torch.cuda.synchronize()
    dt = time.time() - t0
    new_ids = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_ids, skip_special_tokens=True), dt, max_new_tokens/dt

@torch.no_grad()
def wikitext_ppl(model, tok, seq_len=512, max_chunks=20, stride=512):
    """Light WikiText-2 perplexity probe (fast, indicative)."""
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(t for t in ds["text"][:400] if t.strip())
    enc = tok(text, return_tensors="pt").input_ids.to(model.device)
    nll_sum, tok_count = 0.0, 0
    for begin in range(0, enc.size(1) - seq_len, stride):
        chunk = enc[:, begin:begin+seq_len]
        out = model(chunk, labels=chunk)
        nll_sum += out.loss.float().item() * seq_len
        tok_count += seq_len
        if tok_count // seq_len >= max_chunks: break
    return math.exp(nll_sum / tok_count)

results = {}
PROMPT = ("<|im_start|>user\nIn two sentences, explain why post-training "
          "quantization works for large language models.<|im_end|>\n"
          "<|im_start|>assistant\n")

def benchmark(label, model_path_or_id):
    free_mem()
    print(f"\n──── benchmarking: {label} ────")
    tok = AutoTokenizer.from_pretrained(model_path_or_id)
    m = AutoModelForCausalLM.from_pretrained(
            model_path_or_id, torch_dtype="auto", device_map="cuda").eval()
    sample, dt, tps = time_generation(m, tok, PROMPT)
    ppl = wikitext_ppl(m, tok)
    size = dir_size_gb(model_path_or_id) if os.path.isdir(str(model_path_or_id)) else None
    results[label] = {"size_gb": size, "ppl": round(ppl, 3),
                      "latency_s": round(dt, 3), "tok_per_s": round(tps, 1),
                      "sample": sample.strip().replace("\n", " ")[:180]}
    print(json.dumps(results[label], indent=2))
    del m; free_mem()

In [ ]:
print("\n════════════ Baseline (FP16) ════════════")
benchmark("00_fp16_baseline", MODEL_ID)

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

print("\n════════════ Recipe 1: FP8_DYNAMIC ════════════")
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto")
tok = AutoTokenizer.from_pretrained(MODEL_ID)

recipe_fp8 = QuantizationModifier(
    targets="Linear",
    scheme="FP8_DYNAMIC",
    ignore=["lm_head"],
)
oneshot(model=model, recipe=recipe_fp8)

FP8_DIR = "Qwen2.5-0.5B-FP8-Dynamic"
model.save_pretrained(FP8_DIR, save_compressed=True)
tok.save_pretrained(FP8_DIR)
del model; free_mem()
benchmark("01_fp8_dynamic", FP8_DIR)

In [ ]:
NUM_CALIB_SAMPLES = 256
MAX_SEQ_LEN       = 1024

tok = AutoTokenizer.from_pretrained(MODEL_ID)
raw = load_dataset("HuggingFaceH4/ultrachat_200k",
                   split=f"train_sft[:{NUM_CALIB_SAMPLES}]")

def to_text(ex):
    return {"text": tok.apply_chat_template(ex["messages"], tokenize=False)}

def tokenize(ex):
    return tok(ex["text"], padding=False, truncation=True,
               max_length=MAX_SEQ_LEN, add_special_tokens=False)

calib_ds = (raw.shuffle(seed=42)
               .map(to_text)
               .map(tokenize, remove_columns=raw.column_names))
print("Calibration set:", len(calib_ds), "samples, max_seq_len =", MAX_SEQ_LEN)

In [ ]:
from llmcompressor.modifiers.quantization import GPTQModifier

print("\n════════════ Recipe 2: GPTQ W4A16 ════════════")
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto")

recipe_w4a16 = GPTQModifier(
    targets="Linear",
    scheme="W4A16",
    ignore=["lm_head"],
    dampening_frac=0.01,
)
oneshot(
    model=model,
    dataset=calib_ds,
    recipe=recipe_w4a16,
    max_seq_length=MAX_SEQ_LEN,
    num_calibration_samples=NUM_CALIB_SAMPLES,
)

W4A16_DIR = "Qwen2.5-0.5B-W4A16-G128"
model.save_pretrained(W4A16_DIR, save_compressed=True)
tok.save_pretrained(W4A16_DIR)
del model; free_mem()
benchmark("02_gptq_w4a16", W4A16_DIR)

In [ ]:
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier

print("\n════════════ Recipe 3: SmoothQuant + GPTQ W8A8 ════════════")
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto")

recipe_w8a8 = [
    SmoothQuantModifier(smoothing_strength=0.8),
    GPTQModifier(targets="Linear", scheme="W8A8", ignore=["lm_head"]),
]
oneshot(
    model=model,
    dataset=calib_ds,
    recipe=recipe_w8a8,
    max_seq_length=MAX_SEQ_LEN,
    num_calibration_samples=NUM_CALIB_SAMPLES,
)

W8A8_DIR = "Qwen2.5-0.5B-W8A8-SmoothQuant"
model.save_pretrained(W8A8_DIR, save_compressed=True)
tok.save_pretrained(W8A8_DIR)
del model; free_mem()
benchmark("03_smoothquant_w8a8", W8A8_DIR)

print("\n══════════════════════ FINAL SUMMARY ══════════════════════")
print(f"{'Variant':<26}{'Size GB':>9}{'PPL':>10}{'tok/s':>9}{'Latency':>11}")
print("-" * 65)
for k, v in results.items():
    size = f"{v['size_gb']:.3f}" if v['size_gb'] else "  (hub) "
    print(f"{k:<26}{size:>9}{v['ppl']:>10.2f}{v['tok_per_s']:>9.1f}"
          f"{v['latency_s']:>10.2f}s")

print("\nSample completions (greedy, 64 new tokens):")
for k, v in results.items():
    print(f"\n[{k}]\n  → {v['sample']}")

GPU: Tesla T4 | CUDA: 12.8 | torch: 2.10.0+cu128

════════════ Baseline (FP16) ════════════

──── benchmarking: 00_fp16_baseline ────


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

2026-05-15T09:15:07.140112+0000 | _apply_smoothing | INFO - Smoothing with model.layers.8.input_layernorm
2026-05-15T09:15:07.155415+0000 | _apply_smoothing | INFO - Smoothing with model.layers.8.post_attention_layernorm



(10/25): Calibrating: 100%|██████████| 256/256 [00:04<00:00, 60.71it/s]

2026-05-15T09:15:15.671278+0000 | _apply_smoothing | INFO - Smoothing with model.layers.9.input_layernorm


2026-05-15T09:15:15.687491+0000 | _apply_smoothing | INFO - Smoothing with model.layers.9.post_attention_layernorm


(10/25): Propagating:  43%|████▎     | 110/256 [00:01<00:02, 60.82it/s]